# 00 - Architecture overview

This notebook documents the project scaffold and the canonical data flow used by the pricer.

## Canonical flow

```text
External files -> data/raw -> normalized loaders -> data/interim -> pricing/risk -> data/processed -> notebooks/dashboard
```

The notebook stays thin on purpose: all reusable logic belongs to `src/`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import ProjectConfig
from src.factory import ProductFactoryRegistry
from src.market import MarketData
from src.risk import RiskSnapshot


In [2]:
config = ProjectConfig.default(project_root)
config.ensure_directories()

pd.Series(
    {
        "project_root": str(config.project_root),
        "raw_dir": str(config.raw_dir),
        "interim_dir": str(config.interim_dir),
        "processed_dir": str(config.processed_dir),
        "rate_curves_source": str(config.rate_curves_source),
        "options_source": str(config.options_source),
        "inventory_source": str(config.inventory_source),
    },
    name="project_paths",
)


project_root          C:\Users\alixg\OneDrive - Université Paris-Dau...
raw_dir               C:\Users\alixg\OneDrive - Université Paris-Dau...
interim_dir           C:\Users\alixg\OneDrive - Université Paris-Dau...
processed_dir         C:\Users\alixg\OneDrive - Université Paris-Dau...
rate_curves_source    C:\Users\alixg\OneDrive - Université Paris-Dau...
options_source        C:\Users\alixg\OneDrive - Université Paris-Dau...
inventory_source      C:\Users\alixg\OneDrive - Université Paris-Dau...
Name: project_paths, dtype: str

In [3]:
flow = pd.DataFrame(
    [
        ("inputs", "rate curves / option chain / inventory workbook", "external source files"),
        ("raw", "staging helpers", "copies in data/raw"),
        ("interim", "market + portfolio loaders", "normalized tables in data/interim"),
        ("pricing", "products + models + factory", "prices and risk metrics"),
        ("processed", "portfolio aggregation", "reporting tables in data/processed"),
        ("presentation", "notebooks + dashboard", "analytics and user views"),
    ],
    columns=["stage", "main_objects", "output"],
)
flow


,stage,main_objects,output
0,inputs,rate curves / option chain / inventory workbook,external source files
1,raw,staging helpers,copies in data/raw
2,interim,market + portfolio loaders,normalized tables in data/interim
3,pricing,products + models + factory,prices and risk metrics
4,processed,portfolio aggregation,reporting tables in data/processed
5,presentation,notebooks + dashboard,analytics and user views


## Input -> product -> model -> price/risk

The example below is intentionally simple. It only illustrates the dependency chain that later pricing modules will implement in detail.

In [4]:
registry = ProductFactoryRegistry()
registry.register(
    "capital_protected_note",
    lambda product_id: {
        "product_id": product_id,
        "legs": ["zero_coupon_bond", "call_option"],
    },
)

product = registry.build("capital_protected_note", product_id="NOTE-001")
market_data = MarketData(spot=100.0, rate=0.02, volatility=0.25)
risk_view = RiskSnapshot(product_id=product["product_id"], price=0.0, metrics={"delta": 0.0, "vega": 0.0})

{
    "product": product,
    "market_data": market_data,
    "risk_view": risk_view,
}


{'product': {'product_id': 'NOTE-001',
  'legs': ['zero_coupon_bond', 'call_option']},
 'market_data': MarketData(spot=100.0, rate=0.02, volatility=0.25, dividend_yield=0.0),
 'risk_view': RiskSnapshot(product_id='NOTE-001', price=0.0, metrics={'delta': 0.0, 'vega': 0.0})}